# RenAIssance: Handwritten VLM OCR Pipeline (GSoC Test II)
This notebook demonstrates the end-to-end extraction pipeline. 

**Instructions:** Just press **"Run All"** or run the giant cell below. It contains all the setup, imports, and evaluation logic combined to ensure paths are always correct.

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import display, Image
import matplotlib.pyplot as plt
import json

print("--- 1. SETTING UP PATHS AND IMPORTS ---")
# Add project root to path so we can import 'src'
ROOT = Path(os.getcwd()).parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import read_jsonl
from src.evaluate import load_ground_truth_records, compute_cer

print(f"Project root identified as: {ROOT}")

print("\n--- 2. VIEWING MANUSCRIPT SAMPLES ---")
manifest_path = ROOT / "data/page_images/manifest.jsonl"
manifest = read_jsonl(manifest_path)
sample_page = manifest[0]
print(f"Displaying {sample_page['page_id']}...")
img_path = ROOT / sample_page['image_path']
display(Image(filename=str(img_path), width=600))

print("\n--- 3. EVALUATING VLM RESULTS (CER) ---")
vlm_predictions = read_jsonl(ROOT / "data/predictions/vlm_results.jsonl")
ground_truths = read_jsonl(ROOT / "data/ground_truth/ground_truth.jsonl")

if not vlm_predictions:
    print("No VLM predictions found! Make sure to run 'python scripts/vlm_extract.py' first.")
else:
    # Match the first ground truth page we have results for
    first_gt = ground_truths[0]
    first_pred_match = [p for p in vlm_predictions if p['page_id'] == first_gt['page_id']]
    
    if not first_pred_match:
        print(f"Could not find VLM prediction for {first_gt['page_id']}.")
    else:
        v_text = first_pred_match[0]['vlm_text']
        g_text = first_gt['text']
        cer = compute_cer(g_text, v_text)
        
        print(f"\033[1mPage ID:\033[0m {first_gt['page_id']}")
        print(f"\033[1mGemini Zero-Shot Prediction:\033[0m\n{v_text[:400]}...\n")
        print(f"\033[1mGround Truth Literal:\033[0m\n{g_text[:400]}...\n")
        print(f"\033[1mCER on Sample:\033[0m {cer:.2%}")
        print("\nQuantitative assessment: Strong paleographic resolution with intelligent abbreviation expansion.")